This notebook begins with:

- A visual search (MobileNetV2),

- Filtering by visual score,

- Then textual refinement (TF-IDF) on only visually similar images.

### Imports and interactive file selector

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../src"))

import pickle
import pandas as pd
import numpy as np
import time
from fetch import download_file_from_space
from image_features_mobilenet import extract_visual_features
from visual_matcher import build_index, search_similar_images
from model import extract_text, find_similar_to_input

from PIL import Image
from IPython.display import display
import io
import ipywidgets as widgets
from IPython.display import display as ipy_display

test_dir = "../../AI_matcher/tests"
available_files = sorted([
    f for f in os.listdir(test_dir)
    if os.path.isfile(os.path.join(test_dir, f)) and not f.startswith(".DS_Store")
])
dropdown = widgets.Dropdown(options=available_files, description="File:")
ipy_display(dropdown)

### Parameters

In [ ]:
query_path = os.path.join(test_dir, dropdown.value)
image_vector_path = "../../output/image_vectors.pkl"
top_k_visual = 10
similarity_threshold_visual = 50.0

### Visual similarity search

In [ ]:
with open(query_path, "rb") as f:
    query_content = f.read()
query_vec = extract_visual_features(query_content)

with open(image_vector_path, "rb") as f:
    visual_db = pickle.load(f)

index, keys, vectors = build_index(visual_db, n_neighbors=top_k_visual)
visual_results = search_similar_images(query_vec, index, keys, vectors, top_k=top_k_visual)

### Filter visually similar docs and extract text

In [ ]:
textual_candidates = []
print("\nTop visually similar documents (≥ 50%):")
for file_key, score in visual_results:
    if score >= similarity_threshold_visual:
        try:
            print(f"\n{file_key} — {score:.2f}%")
            content = download_file_from_space(file_key)
            text = extract_text(content, file_key)
            if text.strip():
                textual_candidates.append((file_key, text))
            img = Image.open(io.BytesIO(content))
            display(img)
        except Exception as e:
            print(f"[ERROR] Failed to process {file_key}: {e}")

### Apply text-based similarity on visual candidates

In [ ]:
if textual_candidates:
    candidate_keys = [x[0] for x in textual_candidates]
    candidate_texts = {k: t for k, t in textual_candidates}

    with open(query_path, "rb") as f:
        query_text = extract_text(f.read(), query_path)

    df_text = find_similar_to_input(query_text, candidate_texts, top_k=5)
    df_text.columns = ["file", "similarity_text_%"]

    print("\nRefined matches after text-based similarity:")
    print(df_text)
else:
    print("\n[INFO] No visual match ≥ 50% found. Skipping textual refinement.")